# Interactive Poisson Quantile Explorer

This notebook lets you interactively explore the quantiles of a Poisson distribution.

You can:
- Set the mean (λ) of the Poisson distribution.
- Enter a list of quantile probabilities (e.g., [0.5, 0.6, 0.8]).
- See the corresponding quantile values (inverse CDFs).
- Visualize the distribution with quantile markers.

Use the widgets below to experiment and understand how the distribution and quantiles behave.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [4]:
def plot_poisson_quantiles(lambda_poisson, quantile_input):
    clear_output(wait=True)
    
    # Parse quantile input safely
    try:
        quantile_probs = eval(quantile_input)
        if not isinstance(quantile_probs, (list, tuple)):
            raise ValueError
        quantile_probs = [float(p) for p in quantile_probs if 0 <= p <= 1]
    except:
        display(Markdown("**⚠️ Invalid input. Please enter a list of numbers between 0 and 1.**"))
        return

    # Calculate quantiles (inverse CDF)
    quantile_values = poisson.ppf(quantile_probs, mu=lambda_poisson).astype(int)

    # Display the quantile results
    quantile_info = "\n".join([f"P = {p:.2f} → x = {x}" for p, x in zip(quantile_probs, quantile_values)])
    display(Markdown(f"### Quantiles (Inverse CDFs)\n```\n{quantile_info}\n```"))

    # Range for plotting
    x_vals = np.arange(0, poisson.ppf(0.999, mu=lambda_poisson) + 1)
    pmf_vals = poisson.pmf(x_vals, mu=lambda_poisson)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(x_vals, pmf_vals, color='lightblue', edgecolor='black', label='Poisson PMF')

    for q, p in zip(quantile_values, quantile_probs):
        plt.axvline(x=q, color='red', linestyle='--', alpha=0.7)
        plt.text(q + 0.1, poisson.pmf(q, mu=lambda_poisson) + 0.005,
                 f'P={p:.2f}, x={q}', color='red')

    plt.title(f'Poisson Distribution (λ = {lambda_poisson}) with Quantiles')
    plt.xlabel('k')
    plt.ylabel('P(X = k)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [5]:
lambda_slider = widgets.IntSlider(value=5, min=1, max=30, step=1, description='λ (mean):')
quantile_text = widgets.Text(value='[0.5, 0.6, 0.7, 0.8]', description='Quantiles:')

ui = widgets.VBox([lambda_slider, quantile_text])
out = widgets.interactive_output(plot_poisson_quantiles, {
    'lambda_poisson': lambda_slider,
    'quantile_input': quantile_text
})

display(ui, out)